# NBA Player-Season Three-Point Shooting, Fatigue, and Arena Context Measurements, 2018–2023 Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This notebook follows a step-by-step workflow:
- load the Croissant metadata
- inspect the dataset structure using `@id` values
- discover record sets and fields dynamically
- extract records into pandas DataFrames
- perform basic data cleaning and exploratory analysis
- visualize a few useful distributions and relationships

All dataset entities are referenced by their `@id` values whenever available.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import json
import math
import warnings

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/frontiers.j06e-6z0z/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

In [ ]:
# Convert metadata to a JSON-like dictionary for structured inspection
metadata_json = metadata.to_json()

summary_keys = [
    '@id', 'name', 'description', 'identifier', 'version', 'datePublished',
    'license', 'temporalCoverage', 'spatialCoverage', 'keywords'
]

summary = {k: metadata_json.get(k) for k in summary_keys if k in metadata_json}
summary

## 2. Data Overview
Review available record sets, fields, and their IDs.

Because Croissant datasets may expose several record sets, we first inspect the metadata structure dynamically. If the published schema contains record sets, we list each record set `@id` and its field `@id`s. If no record sets are explicitly declared, we note that and still inspect the metadata payload.

In [ ]:
# Inspect top-level metadata keys
print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

record_sets_meta = metadata_json.get('recordSet', [])
print(f"\nNumber of record sets declared in metadata: {len(record_sets_meta)}")

if record_sets_meta:
    for rs in record_sets_meta:
        rs_id = rs.get('@id', '<missing @id>')
        rs_name = rs.get('name', '<unnamed>')
        print(f"\nRecord set @id: {rs_id}")
        print(f"Record set name: {rs_name}")
        fields = rs.get('field', [])
        print(f"Fields: {len(fields)}")
        for field in fields:
            field_id = field.get('@id', '<missing @id>')
            field_name = field.get('name', '<unnamed>')
            data_type = field.get('dataType', '<unknown>')
            source = field.get('source', {})
            column = source.get('column', None) if isinstance(source, dict) else None
            print(f"  - field @id: {field_id} | name: {field_name} | dataType: {data_type} | column: {column}")
else:
    print('No explicit record sets are declared in the Croissant metadata.')
    print('This means extraction must be handled defensively, based on discovered records or distributions.')

In [ ]:
# Show a compact metadata profile helpful for exploration
profile = {
    'dataset_@id': metadata_json.get('@id'),
    'distribution_count': len(metadata_json.get('distribution', [])),
    'record_set_count': len(metadata_json.get('recordSet', [])),
    'author_count': len(metadata_json.get('author', [])) if isinstance(metadata_json.get('author', []), list) else 0,
    'citation_count': len(metadata_json.get('citation', [])) if isinstance(metadata_json.get('citation', []), list) else 0,
    'visualization_count': len(metadata_json.get('visualization', [])) if isinstance(metadata_json.get('visualization', []), list) else 0,
}
profile

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use discovered record set and field `@id`s where available.

This notebook uses variables so the same code works whether the dataset has one or many record sets. If no explicit record set is published, the notebook falls back to metadata-driven inspection and attempts safe extraction.

In [ ]:
# Build a list of record set @id values from metadata
record_sets = [rs.get('@id') for rs in record_sets_meta if rs.get('@id')]
print('Discovered record set @id values:')
print(record_sets)

dataframes = {}
record_samples = {}

if record_sets:
    for record_set_id in record_sets:
        try:
            records = list(dataset.records(record_set=record_set_id))
            record_samples[record_set_id] = records[:3]
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded record set {record_set_id} with shape {dataframes[record_set_id].shape}")
        except Exception as e:
            print(f"Could not load record set {record_set_id}: {e}")
else:
    print('No record set @id values available to pass into dataset.records(record_set=...).')
    print('Skipping direct record extraction from record sets.')

In [ ]:
# Preview extracted DataFrames, if any were loaded
if dataframes:
    first_record_set_id = next(iter(dataframes))
    print(f"First loaded record set @id: {first_record_set_id}")
    print('Columns:')
    print(dataframes[first_record_set_id].columns.tolist())
    display(dataframes[first_record_set_id].head())
else:
    print('No record DataFrames were extracted from explicit record sets.')
    print('We will continue using metadata-derived tables for exploration.')

In [ ]:
# Create metadata-derived DataFrames so the notebook remains fully explorable
dataset_info_df = pd.DataFrame([
    {
        'dataset_@id': metadata_json.get('@id'),
        'name': metadata_json.get('name'),
        'identifier': metadata_json.get('identifier'),
        'version': metadata_json.get('version'),
        'datePublished': metadata_json.get('datePublished'),
        'license': metadata_json.get('license'),
        'temporalCoverage': metadata_json.get('temporalCoverage'),
        'spatialCoverage': metadata_json.get('spatialCoverage'),
        'description': metadata_json.get('description')
    }
])

keywords_df = pd.DataFrame({'keyword': metadata_json.get('keywords', [])})
data_biases_df = pd.DataFrame({'dataBias': metadata_json.get('dataBiases', [])})
data_limitations_df = pd.DataFrame({'dataLimitation': metadata_json.get('dataLimitations', [])})
data_use_cases_df = pd.DataFrame({'dataUseCase': metadata_json.get('dataUseCases', [])})
preprocessing_steps_df = pd.DataFrame({'preprocessingStep': metadata_json.get('dataPreprocessingProtocol', [])})
collection_types_df = pd.DataFrame({'dataCollectionType': metadata_json.get('dataCollectionType', [])})

distribution_df = pd.DataFrame(metadata_json.get('distribution', [])) if isinstance(metadata_json.get('distribution', []), list) else pd.DataFrame()
authors_df = pd.DataFrame(metadata_json.get('author', [])) if isinstance(metadata_json.get('author', []), list) else pd.DataFrame()
citations_df = pd.DataFrame(metadata_json.get('citation', [])) if isinstance(metadata_json.get('citation', []), list) else pd.DataFrame()

metadata_tables = {
    'dataset_info': dataset_info_df,
    'keywords': keywords_df,
    'data_biases': data_biases_df,
    'data_limitations': data_limitations_df,
    'data_use_cases': data_use_cases_df,
    'preprocessing_steps': preprocessing_steps_df,
    'collection_types': collection_types_df,
    'distribution': distribution_df,
    'authors': authors_df,
    'citations': citations_df,
}

for table_name, df in metadata_tables.items():
    print(f"{table_name}: {df.shape}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Since the published Croissant metadata for this dataset does not explicitly list record sets, this section demonstrates two complementary EDA paths:
1. record-level analysis if record sets are available at runtime, and
2. metadata-level analysis that is always available from the Croissant schema.

The code below dynamically selects usable fields instead of assuming a fixed schema.

In [ ]:
# Helper utilities for dynamic EDA
def coerce_numeric_columns(df):
    out = df.copy()
    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors='ignore')
    return out

def find_candidate_numeric_column(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        return numeric_cols[0]
    return None

def find_candidate_group_column(df):
    object_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if object_cols:
        return object_cols[0]
    return None

In [ ]:
# EDA on record-level data if available
if dataframes:
    analysis_record_set_id = next(iter(dataframes))
    df = coerce_numeric_columns(dataframes[analysis_record_set_id])
    print(f"Using record set @id for EDA: {analysis_record_set_id}")
    print(f"Shape: {df.shape}")
    
    numeric_field = find_candidate_numeric_column(df)
    group_field = find_candidate_group_column(df)
    
    print(f"Candidate numeric field @id/column: {numeric_field}")
    print(f"Candidate group field @id/column: {group_field}")
    
    if numeric_field is not None:
        threshold = df[numeric_field].median() if df[numeric_field].notna().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
        
        std_value = filtered_df[numeric_field].std()
        if pd.notna(std_value) and std_value != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
        else:
            filtered_df[f"{numeric_field}_normalized"] = 0.0
        
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        if group_field is not None and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).to_frame(name=f"mean_{numeric_field}")
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print('No numeric fields were detected in the extracted record-level DataFrame.')
else:
    print('Skipping record-level EDA because no record sets were extracted.')

In [ ]:
# Metadata-level EDA: keyword lengths and category summaries
keywords_analysis_df = keywords_df.copy()
if not keywords_analysis_df.empty:
    keywords_analysis_df['keyword_length'] = keywords_analysis_df['keyword'].astype(str).str.len()
    numeric_field = 'keyword_length'
    threshold = 10
    filtered_df = keywords_analysis_df[keywords_analysis_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    
    std_value = filtered_df[numeric_field].std()
    if pd.notna(std_value) and std_value != 0:
        filtered_df[f'{numeric_field}_normalized'] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
        )
    else:
        filtered_df[f'{numeric_field}_normalized'] = 0.0
    
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())
    
    group_field = 'keyword_length'
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field).size().to_frame('count')
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())

In [ ]:
# Additional useful summaries from the metadata-derived tables
print('Collection types:')
display(collection_types_df)

print('Preprocessing steps:')
display(preprocessing_steps_df)

print('Data limitations:')
display(data_limitations_df)

print('Data biases:')
display(data_biases_df)

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The plots below focus on information that is guaranteed to be available from the Croissant metadata, with optional record-level plots if data records are exposed.

In [ ]:
# Visualization 1: keyword length distribution
if not keywords_df.empty:
    plot_df = keywords_df.copy()
    plot_df['keyword_length'] = plot_df['keyword'].astype(str).str.len()
    
    plt.figure(figsize=(10, 5))
    sns.histplot(plot_df['keyword_length'], bins=10, kde=True)
    plt.title('Distribution of Keyword Lengths')
    plt.xlabel('Keyword length')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('No keywords available for visualization.')

In [ ]:
# Visualization 2: counts of metadata categories
category_counts = pd.DataFrame({
    'category': ['keywords', 'data_biases', 'data_limitations', 'data_use_cases', 'preprocessing_steps', 'collection_types', 'distributions'],
    'count': [
        len(keywords_df),
        len(data_biases_df),
        len(data_limitations_df),
        len(data_use_cases_df),
        len(preprocessing_steps_df),
        len(collection_types_df),
        len(distribution_df),
    ]
})

plt.figure(figsize=(11, 5))
sns.barplot(data=category_counts, x='category', y='count', palette='viridis')
plt.title('Counts of Dataset Metadata Components')
plt.xlabel('Metadata category')
plt.ylabel('Count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

display(category_counts)

In [ ]:
# Optional record-level visualization if extracted data are available
if dataframes:
    analysis_record_set_id = next(iter(dataframes))
    df = coerce_numeric_columns(dataframes[analysis_record_set_id])
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    object_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if len(numeric_cols) >= 1:
        plt.figure(figsize=(10, 5))
        sns.histplot(df[numeric_cols[0]].dropna(), kde=True)
        plt.title(f'Distribution of {numeric_cols[0]} in record set {analysis_record_set_id}')
        plt.xlabel(numeric_cols[0])
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()
    
    if len(numeric_cols) >= 1 and len(object_cols) >= 1:
        grouped_plot_df = (
            df[[object_cols[0], numeric_cols[0]]]
            .dropna()
            .groupby(object_cols[0])[numeric_cols[0]]
            .mean()
            .sort_values(ascending=False)
            .head(10)
            .reset_index()
        )
        
        plt.figure(figsize=(11, 5))
        sns.barplot(data=grouped_plot_df, x=object_cols[0], y=numeric_cols[0], palette='magma')
        plt.title(f'Mean {numeric_cols[0]} by {object_cols[0]}')
        plt.xlabel(object_cols[0])
        plt.ylabel(f'Mean {numeric_cols[0]}')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print('Skipping optional record-level plots because no record set data were extracted.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This Croissant dataset describes an NBA player-season dataset covering 2018–2023, with emphasis on three-point shooting, fatigue indicators, and arena context.

Key observations from the metadata exploration:
- the dataset is well documented with rich descriptive metadata
- multiple data sources were integrated, including tracking, trajectory, manual charting, wearables, and arena logs
- the metadata explicitly documents preprocessing, limitations, biases, and use cases
- the published Croissant schema currently does not expose explicit `recordSet` entries in the metadata snapshot, so record-level loading may depend on future schema revisions or alternate distribution access

If explicit record sets become available, the same notebook structure can be reused by referencing each record set and field by its `@id` and extending the record-level EDA sections.